In [5]:
from IPython.display import Video, display
import cv2
import numpy as np

In [6]:
in_path  = "videos/group_11_fixed.mp4"       
out_path = "outputs/group_11_fixed_test.mp4"

In [7]:
#display(Video(in_path))

In [8]:

cap = cv2.VideoCapture(in_path)
if not cap.isOpened():
    raise IOError(f"Could not open {in_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v") 
writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

ok, first_frame = cap.read()
if not ok:
    print("no frames")

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    # big cube
    #frame_bgr[200:250, 75:250, :] = 0

    # negative
    #neg = 255 - frame_bgr

    # ~ is "not" et 
    #green_mask = ~(frame_bgr[:, :, 0] > 150) & (frame_bgr[:, :, 1] > 60) & ~(frame_bgr[:, :, 2] > 120)
    #frame_bgr[green_mask] = [0, 0, 0]

    # green converter

    # RGB to HSV
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)

    # HSV range for green
    lower_green = np.array([30, 40, 40])
    upper_green = np.array([75, 255, 255])

    # mask where green = 255 and 0 otherwise
    keep_green_mask = cv2.inRange(hsv, lower_green, upper_green)
    # mask where green = 0 and 255 otherwise
    no_green_mask = cv2.bitwise_not(keep_green_mask)

    # use the no green mask
    no_green_frame = cv2.bitwise_and(frame_bgr, frame_bgr, mask=no_green_mask)

    # vreate a uniform color frame
    fixed_color_img = np.zeros_like(frame_bgr)
    fixed_color_img[:] = [0, 0, 255]

    # keep only where we had green originaly
    green_to_color = cv2.bitwise_and(fixed_color_img, fixed_color_img, mask=keep_green_mask)
    green_to_first_frame = cv2.bitwise_and(first_frame, first_frame, mask=keep_green_mask)

    replace_green = green_to_first_frame

    # add the 2 images
    output_frame = cv2.add(no_green_frame, replace_green)

    writer.write(output_frame)

cap.release()
writer.release()
print("Saved:", out_path)

display(Video(out_path))

Saved: outputs/group_11_fixed_test.mp4
